# 05 - Ablations: Forecast Value (1-5)

**Purpose.** Compare predictive-MCDM against five forecast-variant baselines per PROJECT_2_PLAN.md S10:

1. No-forecast EMA baseline (Baseline C)
2. Naive last-observation forecast
3. CIR-calibrated forecast (Baseline D)
4. Markov-switching short-rate (2-state)
5. CatBoost on hand-crafted features

Each section is a stub the user fills in by routing the forecaster object through `MCDMEMAStrategy` (with the rate replaced).

**Expected runtime.** 2-4 minutes per forecaster on synthetic data.


In [ ]:
# --- path preamble: make sibling packages importable ---
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "extras" / "fractal_pr_lending_allocation"))

print(f"ROOT = {ROOT}")


In [ ]:
# Synthetic-data fallback: mirrors forecaster.train._make_synth_df.
# Used whenever the real joined_clean.parquet is not yet on disk.
import numpy as np
import pandas as pd


def make_synth_joined(n_rows: int = 2000, seed: int = 0) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2025-01-01", periods=n_rows, freq="h", tz="UTC")

    def util_walk(start: float) -> np.ndarray:
        u = np.empty(n_rows)
        u[0] = start
        for i in range(1, n_rows):
            u[i] = np.clip(u[i - 1] + rng.normal(0.0, 0.01), 0.05, 0.97)
        return u

    u_a = util_walk(0.55)
    u_c = util_walk(0.45)

    # Toy rate process: kink-shaped baseline plus Gaussian residual.
    r_a = 0.05 * (u_a / 0.92) * 0.90 * u_a + rng.normal(0, 0.002, n_rows)
    r_c = 0.04 * u_c + rng.normal(0, 0.002, n_rows)
    r_a = np.clip(r_a, 0.0, 0.5)
    r_c = np.clip(r_c, 0.0, 0.5)

    tvl_a = 1e8 + np.cumsum(rng.normal(0, 1e5, n_rows))
    tvl_c = 5e7 + np.cumsum(rng.normal(0, 5e4, n_rows))
    gas = np.clip(
        20 + 5 * rng.standard_normal(n_rows) + 10 * np.sin(np.arange(n_rows) / 24),
        5, 200,
    )
    eth = 3000 + np.cumsum(rng.normal(0, 5, n_rows))

    return pd.DataFrame({
        "r_aave": r_a, "r_compound": r_c,
        "u_aave": u_a, "u_compound": u_c,
        "tvl_aave": tvl_a, "tvl_compound": tvl_c,
        "gas_gwei": gas, "eth_usd": eth,
    }, index=idx)


def load_joined(path: str = "data/cached/joined_clean.parquet") -> tuple[pd.DataFrame, bool]:
    """Try the real cached panel; fall back to synthetic on FileNotFoundError."""
    full = ROOT / path
    try:
        df = pd.read_parquet(full)
        print(f"[real] loaded {len(df):,} rows from {full}")
        return df, True
    except FileNotFoundError:
        print(f"[synth] {full} not found - generating synthetic panel")
        return make_synth_joined(), False


df, is_real = load_joined()
df.head()


## Ablation 1 - EMA baseline (Baseline C; mandatory strawman)

In [ ]:
from strategies.baseline_mcdm_ema import MCDMEMAStrategy, MCDMEMAParams
from backtest.observations_builder import build_all

_, (_, val_obs, test_obs) = build_all(synthetic=True)
params = MCDMEMAParams(INITIAL_BALANCE=1_000_000.0, DEFAULT_INITIAL_ENTITY='AAVE')
strat = MCDMEMAStrategy(debug=False, params=params)
res_ema = strat.run(test_obs)
print('EMA baseline:', res_ema.get_default_metrics())


## Ablation 2 - Naive last-observation forecast

In [ ]:
# Stub: replace EMA(alpha=0.3) with EMA(alpha=1.0) ~ identity to get
# the naive last-observation forecast. (Both Aave and Compound get
# their realised rate as the forecast.)
params_naive = MCDMEMAParams(INITIAL_BALANCE=1_000_000.0,
                              DEFAULT_INITIAL_ENTITY='AAVE',
                              EMA_ALPHA=1.0)
strat = MCDMEMAStrategy(debug=False, params=params_naive)
res_naive = strat.run(test_obs)
print('Naive last-obs:', res_naive.get_default_metrics())


## Ablation 3 - CIR-calibrated forecast (Baseline D)

In [ ]:
import numpy as np
from forecaster.baseline_cir import CIRForecaster

cir = CIRForecaster(partitioning='data-driven', n_partitions=3)
cir.fit(df['r_aave'], utilization_series=df['u_aave'])
r_hat = cir.predict(df['r_aave'].iloc[-200:].values, horizon=12)
print(f'CIR Aave 12h forecast: {r_hat:.4%}')

# TODO: wrap CIRForecaster in a strategy adapter (see PROJECT_2_PLAN.md
# Week 2 Day Thu 28: strategies/baseline_mcdm_cir.py).


## Ablation 4 - Markov-switching short-rate (2-state on utilization)

In [ ]:
try:
    from forecaster.baseline_markov import MarkovSwitchingForecaster
    msf = MarkovSwitchingForecaster(n_states=2)
    msf.fit(df['r_aave'], df['u_aave'])
    r_hat = msf.predict(df['r_aave'].iloc[-200:].values, horizon=12)
    print(f'Markov-switching 12h forecast: {r_hat:.4%}')
except (ImportError, AttributeError) as e:
    print(f'MarkovSwitchingForecaster not yet available ({e}).')
    print('See forecaster/baseline_markov.py for implementation status.')


## Ablation 5 - CatBoost on hand-crafted features

In [ ]:
from forecaster.baseline_catboost import CatBoostForecaster, CatBoostConfig

cb_cfg = CatBoostConfig(iterations=200, depth=4)
cb = CatBoostForecaster(cb_cfg, horizon=12)

n_cut = int(0.7 * len(df))
cb.fit(df.iloc[:n_cut], val_df=df.iloc[n_cut:])
print('CatBoost trained on', n_cut, 'rows; feature count:', len(cb.feature_names_))


## Roll-up: forecast quality table

In [ ]:
import pandas as pd
rows = []
for name, res in [('EMA', res_ema), ('Naive', res_naive)]:
    m = res.get_default_metrics()
    rows.append({'forecaster': name,
                 'apy': getattr(m, 'apy', None),
                 'sharpe': getattr(m, 'sharpe', None),
                 'max_dd': getattr(m, 'max_drawdown', None)})
rollup = pd.DataFrame(rows)
rollup


## Next steps

- Add CIR and CatBoost as full strategy adapters (Week 2 Day Thu 28 /   Week 3 Day Tue 2 in the plan).
- Compute per-protocol R^2 and directional accuracy (PROJECT_2_PLAN.md S9.4)   to populate the headline forecaster comparison table.

Relevant plan section: **PROJECT_2_PLAN.md S10, Ablations 1-5.**
